In [3]:
#Tải các thư viện Python dùng cho việc xử lý dữ liệu
import pandas as pd
import numpy as np
print("Đã import thư viện thành công!")

Đã import thư viện thành công!


In [ ]:
# Đọc file dữ liệu gốc từ thư mục data
data_path = "../data/RT_IoT2022"  

df = pd.read_csv(data_path)

# Kiểm tra 5 dòng đầu tiên để xem dữ liệu mặt mũi ra sao
print(f"Kích thước ban đầu của dữ liệu: Dòng = {df.shape[0]}, Cột = {df.shape[1]}")
df.head()

Kích thước ban đầu của dữ liệu: Dòng = 123117, Cột = 85


,Unnamed: 0,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,...,active.std,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,0,38667,1883,tcp,mqtt,32.011598,9,5,3,3,...,0.0,2.972918e+07,2.972918e+07,2.972918e+07,2.972918e+07,0.0,64240,26847,502,MQTT_Publish
1,1,51143,1883,tcp,mqtt,31.883584,9,5,3,3,...,0.0,2.985528e+07,2.985528e+07,2.985528e+07,2.985528e+07,0.0,64240,26847,502,MQTT_Publish
2,2,44761,1883,tcp,mqtt,32.124053,9,5,3,3,...,0.0,2.984215e+07,2.984215e+07,2.984215e+07,2.984215e+07,0.0,64240,26847,502,MQTT_Publish
3,3,60893,1883,tcp,mqtt,31.961063,9,5,3,3,...,0.0,2.991377e+07,2.991377e+07,2.991377e+07,2.991377e+07,0.0,64240,26847,502,MQTT_Publish
4,4,51087,1883,tcp,mqtt,31.902362,9,5,3,3,...,0.0,2.981470e+07,2.981470e+07,2.981470e+07,2.981470e+07,0.0,64240,26847,502,MQTT_Publish


In [ ]:
# Xóa cột 'Unnamed: 0' nếu nó xuất hiện do lúc lưu file bị thừa index cũ
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
    print("Đã xóa cột 'Unnamed: 0' thừa.")
else:
    print("Không tìm thấy cột 'Unnamed: 0', dữ liệu sạch sẽ.")

Đã xóa cột 'Unnamed: 0' thừa.


In [ ]:
# Kiểm tra và xóa các dòng dữ liệu bị trùng lặp hoàn toàn
print(f"Số dòng trước khi xóa trùng: {df.shape[0]}")

df = df.drop_duplicates()

print(f"Số dòng sau khi xóa trùng: {df.shape[0]}")

Số dòng trước khi xóa trùng: 123117
Số dòng sau khi xóa trùng: 117922


In [17]:
# Đếm lại số mẫu từng nhãn sau khi xóa trùng để biết lớp nào bị giảm
print(df['Attack_type'].value_counts()) 

Attack_type
2     90089
10     7654
0      7625
3      4142
8      2584
9      2010
6      2000
7      1002
1       533
11      219
4        36
5        28
Name: count, dtype: int64


In [ ]:
# Xóa cột 'bwd_URG_flag_count' vì nó là cột hằng số (chỉ có 1 giá trị duy nhất, không có ích cho phân loại)
constant_column = 'bwd_URG_flag_count'


if constant_column in df.columns:
    df = df.drop(columns=[constant_column])
    print(f"Đã xóa thành công cột hằng số: {constant_column}")
else:
    print(f"Cột {constant_column} không tồn tại trong DataFrame.")

Đã xóa thành công cột hằng số: bwd_URG_flag_count


In [ ]:
#Lọc và loại bỏ các cột có độ tương quan tuyệt đối > 0.95 với nhau 

# Tính ma trận tương quan cho các biến số
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr().abs()

# Lấy phần tam giác trên của ma trận tương quan
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Tìm tên các cột có hệ số tương quan lớn hơn 0.95
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]

print(f"Các cột có tương quan > 0.95 sẽ bị loại bỏ ({len(to_drop)} cột):")
print(to_drop)

# Thực hiện xóa khỏi DataFrame chính
df = df.drop(columns=to_drop)
print(f"Số cột còn lại sau khi loại bỏ đa cộng tuyến: {df.shape[1]}")

Các cột có tương quan > 0.95 sẽ bị loại bỏ (24 cột):
['bwd_data_pkts_tot', 'bwd_pkts_per_sec', 'flow_pkts_per_sec', 'bwd_header_size_tot', 'bwd_header_size_max', 'flow_ACK_flag_count', 'bwd_pkts_payload.tot', 'bwd_pkts_payload.std', 'flow_pkts_payload.max', 'flow_pkts_payload.tot', 'fwd_iat.tot', 'fwd_iat.std', 'flow_iat.max', 'flow_iat.tot', 'flow_iat.std', 'payload_bytes_per_second', 'bwd_subflow_bytes', 'fwd_bulk_packets', 'bwd_bulk_packets', 'active.std', 'idle.min', 'idle.max', 'idle.tot', 'idle.avg']
Số cột còn lại sau khi loại bỏ đa cộng tuyến: 59


In [ ]:
#Kiểm tra xem dữ liệu còn ô nào bị trống (NaN/Null) không
missing_count = df.isnull().sum().sum()
print(f"Tổng số giá trị bị thiếu trong toàn bộ dataset: {missing_count}")

# Nếu có giá trị thiếu, ta có thể điền giá trị 0 hoặc xóa đi (ở đây RT-IoT2022 không có missing value)

Tổng số giá trị bị thiếu trong toàn bộ dataset: 0


In [ ]:
#Chuyển đổi nhãn dạng chữ (Attack_type) sang dạng số nguyên để thuật toán máy học hiểu được
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

# Kiểm tra tên cột chứa nhãn nhắm tới (thường là 'Attack_type' hoặc tương tự)
target_column = 'Attack_type' 

if target_column in df.columns:
    df[target_column] = encoder.fit_transform(df[target_column])
    print(f"Đã mã hóa thành công cột mục tiêu '{target_column}' sang dạng số.")
    
    # In ra vài ánh xạ nhãn để dễ theo dõi
    class_mapping = dict(zip(encoder.classes_, range(len(encoder.classes_))))
    print("Ví dụ một vài nhãn sau khi mã hóa:", list(class_mapping.items())[:5])
else:
    print(f"Không tìm thấy cột mục tiêu {target_column}, hãy kiểm tra lại tên cột!")

Đã mã hóa thành công cột mục tiêu 'Attack_type' sang dạng số.
Ví dụ một vài nhãn sau khi mã hóa: [(np.int64(0), 0), (np.int64(1), 1), (np.int64(2), 2), (np.int64(3), 3), (np.int64(4), 4)]


In [21]:
from sklearn.model_selection import train_test_split

# Tách X (đặc trưng) và y (nhãn đã mã hóa số)
X = df.drop(columns=['Attack_type'])
y = df['Attack_type']

# One-Hot cho proto và service: mỗi giá trị thành một cột 0/1, không tạo thứ tự giả
X = pd.get_dummies(X, columns=['proto', 'service'], dtype=int)

# Chia 80% train, 20% test, giữ nguyên tỷ lệ từng nhãn, cố định random_state để chạy lại ra kết quả giống nhau
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# In kích thước và số mẫu mỗi nhãn trong tập test để kiểm tra lớp nhỏ còn đủ mẫu không
print("Train:", X_train.shape, "Test:", X_test.shape)
print(y_test.value_counts())

Train: (94337, 69) Test: (23585, 69)
Attack_type
2     18018
10     1531
0      1525
3       828
8       517
9       402
6       400
7       200
1       107
11       44
4         7
5         6
Name: count, dtype: int64


In [22]:
from sklearn.preprocessing import StandardScaler

# Tạo bộ chuẩn hóa
scaler = StandardScaler()

# Học trung bình và độ lệch chuẩn chỉ trên tập train, rồi chuẩn hóa train
X_train_scaled = scaler.fit_transform(X_train)

# Dùng lại đúng các số đã học ở train để chuẩn hóa test (không fit lại)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Lưu dataframe đã sạch sẽ và xử lý xong vào thư mục data/processed/ để đem đi huấn luyện mô hình
import os

# Tạo thư mục processed nếu chưa có
os.makedirs("../data/processed", exist_ok=True)

output_file = "../data/processed/RT_IoT2022_processed.csv"
df_processed.to_csv(output_file, index=False)

print(f"Đã lưu thành công file dữ liệu sạch tại: {output_file}")
print(f"Kích thước dữ liệu cuối cùng: Dòng = {df_processed.shape[0]}, Cột = {df_processed.shape[1]}")

Đã lưu thành công file dữ liệu sạch tại: ../data/processed/RT_IoT2022_processed.csv
Kích thước dữ liệu cuối cùng: Dòng = 117922, Cột = 59


In [23]:
import os, joblib

# Tạo thư mục lưu nếu chưa có
os.makedirs("../data/processed", exist_ok=True)

# Lưu bản chưa chuẩn hóa, dùng cho cây quyết định và Random Forest
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

# Lưu bộ chuẩn hóa để dùng lại cho KNN, SVM, MLP, Logistic Regression
joblib.dump(scaler, "../data/processed/scaler.pkl")

# Lưu ánh xạ nhãn đầy đủ (tên nhãn tương ứng số) để vẽ Confusion Matrix
pd.Series(encoder.classes_).to_csv("../data/processed/label_names.csv", index=False)